In [1]:
from chore_tracker.routes import routes_bp
from flask import Flask, render_template, request, redirect, session, url_for
from chore_tracker.utils import Config,get_db_connection, ChoreActions, ChoreData, UserActions
from datetime import date
cfg = Config.from_yaml()
conn = get_db_connection()

In [3]:
c = conn.execute('''SELECT * FROM completed_chores''').fetchall()
[dict(row) for row in c]

[]

In [4]:
action = ChoreActions(conn)

Child ID: 4, 
              Earned: 213.25, 
              Behavior Deductions: -28.25, 
              Behavior Increases: 22.0,
              Expenses: -150.0, 
              Net: 57.0
Child ID: 5, 
              Earned: 212.5, 
              Behavior Deductions: -29.25, 
              Behavior Increases: 23.0,
              Expenses: -150.0, 
              Net: 56.25
Child ID: 25, 
              Earned: 170.0, 
              Behavior Deductions: -25.75, 
              Behavior Increases: 23.0,
              Expenses: -150.0, 
              Net: 17.25
Child ID: 4, 
              Earned: 213.25, 
              Behavior Deductions: -28.25, 
              Behavior Increases: 22.0,
              Expenses: -150.0, 
              Net: 57.0
Child ID: 5, 
              Earned: 212.5, 
              Behavior Deductions: -29.25, 
              Behavior Increases: 23.0,
              Expenses: -150.0, 
              Net: 56.25
Child ID: 25, 
              Earned: 170.0, 
              Behavior D

In [6]:
users = UserActions(conn)

In [7]:
users.fetch_user_id(username='Evelyn')

4

In [2]:
data = ChoreData(conn)

Child ID: 4, 
              Earned: 213.25, 
              Behavior Deductions: -28.25, 
              Behavior Increases: 22.0,
              Expenses: -150.0, 
              Net: 57.0
Child ID: 5, 
              Earned: 212.5, 
              Behavior Deductions: -29.25, 
              Behavior Increases: 23.0,
              Expenses: -150.0, 
              Net: 56.25
Child ID: 25, 
              Earned: 170.0, 
              Behavior Deductions: -25.75, 
              Behavior Increases: 23.0,
              Expenses: -150.0, 
              Net: 17.25


In [9]:
from datetime import datetime, timedelta
today = datetime.now().date()  # Get today's date without time
thirty_days_ago = today - timedelta(days=30)  # 30 days ago from today 
completed_chores = conn.execute('''
            SELECT u.name as child_name, c.completion_date, COUNT(*) as count
            FROM completed_chores c
            JOIN users u ON c.user_id = u.id
            WHERE c.completion_date BETWEEN ? AND ?
            GROUP BY u.name, c.completion_date
            ORDER BY u.name, c.completion_date
        ''', (thirty_days_ago, today)).fetchall()  # Note the order: (start_date, end_date)
completed_chores

/tmp/ipykernel_2931/3097284128.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  completed_chores = conn.execute('''


[]

In [10]:
print(data.completed_chores_timeline())


{}


In [7]:
quick_id = action.fetch_choreid('10 Minute Helpfulness','preset','Any')
completion_date = date.today().isoformat()
action.complete_chore(2,quick_id,completion_date)

TypeError: 'NoneType' object is not subscriptable

In [8]:
children = data.fetch_children()
earnings = data.get_earnings_report()
print(earnings)

Child ID: 2, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 3, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
Child ID: 1, Earned: 0, Behavior Deductions: 0, Expenses: 0, Net: 0
[{'name': 'Evelyn', 'net_earnings': 0}, {'name': 'Lucy', 'net_earnings': 0}, {'name': 'Virginia', 'net_earnings': 0}]


In [5]:
action.fetch_chores()

In [9]:
chores = [dict(x) for x in action.conn.execute('SELECT * FROM chores').fetchall()]
chores[:5]

[{'id': 1,
  'name': '20 minute reading',
  'assigned': 'Evelyn',
  'time': 20.0,
  'type': 'school'},
 {'id': 2,
  'name': '20 minute reading',
  'assigned': 'Virginia',
  'time': 20.0,
  'type': 'school'},
 {'id': 3,
  'name': '10 minute reading',
  'assigned': 'Lucy',
  'time': 10.0,
  'type': 'school'},
 {'id': 4,
  'name': '20 minutes of reading',
  'assigned': 'Evelyn',
  'time': 20.0,
  'type': 'Daily'},
 {'id': 5,
  'name': '20 minutes of reading',
  'assigned': 'Lucy',
  'time': 20.0,
  'type': 'Daily'}]

In [10]:
def calculate_minutes_for_earnings(earnings):
    return round((earnings * 60) / cfg.hourly_rate)

In [11]:
action.calculate_minutes_for_earnings(52)

260

In [12]:
data.child_behavior_deductions(2)

0

In [13]:
data.behavior_deductions

[{'name': 'Evelyn', 'behavior_deductions': -3.0},
 {'name': 'Lucy', 'behavior_deductions': -2.25},
 {'name': 'Virginia', 'behavior_deductions': -2.0}]

In [15]:
data = ChoreData(conn)
chores = data.fetch_all_chores()
list(set([chore['name'] for chore in chores]))

Child ID: 4, 
              Earned: 58.25, 
              Behavior Deductions: -3.0, 
              Behavior Increases: 0.5,
              Expenses: -40.0, 
              Net: 15.75
Child ID: 5, 
              Earned: 65.0, 
              Behavior Deductions: -2.25, 
              Behavior Increases: 1.25,
              Expenses: -40.0, 
              Net: 24.0
Child ID: 25, 
              Earned: 18.25, 
              Behavior Deductions: -2.0, 
              Behavior Increases: 1.75,
              Expenses: -50.0, 
              Net: -32.0


['Empty Trash',
 'Get Ready For Bed',
 'Vacuum',
 'Tidy Boot Bench',
 'Wipe Counters',
 'Clear Table',
 '10 minute reading',
 'Walk Dog Evening',
 'Walk Dog',
 'Tidy room',
 'Set Table',
 'Tidy Living Room',
 'Clear Dishwasher',
 '20 minutes of reading',
 'Get Ready For Day',
 '20 minute reading',
 'Clean up after yourself',
 'Feed Dog',
 'Laundry']

In [18]:
users = UserActions(conn)
users_names = users.fetch_user_names()

users_names


['Evelyn', 'Lucy', 'Virginia']

In [5]:
def debug_check_users():
    users = conn.execute("SELECT name, role FROM users").fetchall()
    print(dict(users))
debug_check_users()

{'Evelyn': 'child', 'Lucy': 'child', 'Parent': 'parent', 'Virginia': 'child'}


In [16]:
from datetime import datetime, timedelta
import random
def add_sample_data(conn,user_list,num=120):
    users = UserActions(conn)
    data = ChoreData(conn)
    actions = ChoreActions(conn)
    today = datetime.now().date()  # Get today's date without time
    thirty_days_ago = today - timedelta(days=30)  # 30 days ago from today
    # users.add_user('Parent','parent','password')
    # users.add_user('Virginia','child','virginia')
    # users.add_user('Evelyn','child','evelyn')
    # users.add_user('Lucy','child','lucy')
    chores =  data.fetch_all_chores()
    behaviors = data.fetch_all_behaviors()

    for user in user_list:
        # add user,type,and password to users table
        conn.execute('INSERT OR IGNORE INTO users (name, role, password) VALUES (?, ?, ?)',
                     (user, 'child', 'password'))
        # add 10 random completed chores for each user
        # select 10 random dates from last 30 days
        for date in range(num):
            # select random date from last 30 days
            random_date = today - timedelta(days=random.randint(0, 30))
            random_behavior_date = today - timedelta(days=random.randint(0, 30))
            # select random chore from chores
            random_chore = random.choice(chores)
            # select random amount from 1 to 10
            random_amount = random.randint(1, num)
            behavior = random.choice(behaviors)
            # select random behavior from behavior_deductions
            earnings = actions.calculate_earnings(random_amount)
            #add expenses
            random_expense_date = today - timedelta(days=random.randint(0, 30))
             # fetch user id from users
            user_id = conn.execute('SELECT id FROM users WHERE name = ?', (user,)).fetchone()['id']
            # insert into expenses
            conn.execute('INSERT INTO expenses (user_id,expense,amount,date) VALUES (?, ?, ?, ?)',
                         (user_id,'test_expense',-5,random_expense_date))
            # insert into behaviors
            conn.execute('INSERT INTO behaviors (user_id, behavior, amount, type, date) VALUES (?, ?, ?, ?, ?)',
                         (user_id, behavior['behavior'], behavior['amount'], behavior['type'], random_behavior_date,))
            # fetch chore id from chores
            chore_id = conn.execute('SELECT id FROM chores WHERE name = ?', (random_chore['name'],)).fetchone()['id']
            # insert into completed_chores
            conn.execute('INSERT INTO completed_chores (user_id, chore_id, amount_earned, time, completion_date) VALUES (?, ?, ?, ?, ?)',
                         (user_id, chore_id, earnings, random_amount, random_date,))
            
       
    # commit changes
    conn.commit()

def remove_sample_data(conn):
    users ['Bob', 'Alice', 'Charlie', 'Diana']
    # delete all data from completed_chores from listed users:
    for user in users:
        conn.execute('DELETE FROM completed_chores WHERE user_id = (SELECT id FROM users WHERE name = ?)', (user,))
    # delete all data from chores from listed users
    for user in users:
        conn.execute('DELETE FROM users WHERE name = ?', (user,))
    # delete all data from users
    conn.commit()  # behavior_deductions = data.fetch_behavior_deductions()
            # random_deduction = random.randint(1, num)

In [31]:
add_sample_data(conn,users_names,10)

Child ID: 4, 
              Earned: 204.0, 
              Behavior Deductions: -26.5, 
              Behavior Increases: 21.0,
              Expenses: -150.0, 
              Net: 48.5
Child ID: 5, 
              Earned: 201.25, 
              Behavior Deductions: -26.25, 
              Behavior Increases: 21.25,
              Expenses: -150.0, 
              Net: 46.25
Child ID: 25, 
              Earned: 160.75, 
              Behavior Deductions: -24.75, 
              Behavior Increases: 21.0,
              Expenses: -150.0, 
              Net: 7.0
Child ID: 4, 
              Earned: 204.0, 
              Behavior Deductions: -26.5, 
              Behavior Increases: 21.0,
              Expenses: -150.0, 
              Net: 48.5
Child ID: 5, 
              Earned: 201.25, 
              Behavior Deductions: -26.25, 
              Behavior Increases: 21.25,
              Expenses: -150.0, 
              Net: 46.25
Child ID: 25, 
              Earned: 160.75, 
              Behavior D

/tmp/ipykernel_19835/771888678.py:38: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.execute('INSERT INTO expenses (user_id,expense,amount,date) VALUES (?, ?, ?, ?)',
/tmp/ipykernel_19835/771888678.py:41: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.execute('INSERT INTO behaviors (user_id, behavior, amount, type, date) VALUES (?, ?, ?, ?, ?)',
/tmp/ipykernel_19835/771888678.py:46: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  conn.execute('INSERT INTO completed_chores (user_id, chore_id, amount_earned, time, completion_date) VALUES (?, ?, ?, ?, ?)',


In [11]:
import sqlite3

def execute_sql_file(conn, sql_file_path):
    """
    Executes an SQL file against the SQLite database.

    Args:
        db_path (str): Path to the SQLite database file.
        sql_file_path (str): Path to the SQL file containing DROP TABLE statements.
    """
    try:
    
        
        # Read SQL file
        with open(sql_file_path, 'r') as sql_file:
            sql_script = sql_file.read()
        
        # Execute SQL script
        conn.executescript(sql_script)
        
        # Commit and close connection
        conn.commit()

        print("SQL script executed successfully.")

    except Exception as e:
        print(f"Error executing SQL file: {e}")

# Example usage
execute_sql_file(conn, "./chore_tracker/sql/reset_tables.sql")
execute_sql_file(conn, "./chore_tracker/sql/schema.sql")


SQL script executed successfully.
SQL script executed successfully.


In [14]:
reset_table = 'behaviors'
conn.execute(f'DROP TABLE IF EXISTS {reset_table}')
conn.execute("""CREATE TABLE IF NOT EXISTS behaviors (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    behavior TEXT NOT NULL,
    amount REAL NOT NULL,
    type TEXT NOT NULL,
    date DATE NOT NULL,
    FOREIGN KEY (user_id) REFERENCES users(id));""")
conn.commit()

In [7]:
chores = conn.execute('''SELECT * FROM chores''').fetchall()
[dict(row) for row in chores]

[{'id': 1,
  'name': '20 minute reading',
  'assigned': 'Evelyn',
  'time': 20.0,
  'type': 'school'},
 {'id': 2,
  'name': '20 minute reading',
  'assigned': 'Virginia',
  'time': 20.0,
  'type': 'school'},
 {'id': 3,
  'name': '10 minute reading',
  'assigned': 'Lucy',
  'time': 10.0,
  'type': 'school'},
 {'id': 4,
  'name': '20 minutes of reading',
  'assigned': 'Evelyn',
  'time': 20.0,
  'type': 'Daily'},
 {'id': 5,
  'name': '20 minutes of reading',
  'assigned': 'Lucy',
  'time': 20.0,
  'type': 'Daily'},
 {'id': 6,
  'name': '20 minutes of reading',
  'assigned': 'Virginia',
  'time': 20.0,
  'type': 'Daily'},
 {'id': 7,
  'name': 'Clean up after yourself',
  'assigned': 'Evelyn',
  'time': 1.0,
  'type': 'Daily'},
 {'id': 8,
  'name': 'Clean up after yourself',
  'assigned': 'Lucy',
  'time': 1.0,
  'type': 'Daily'},
 {'id': 9,
  'name': 'Clean up after yourself',
  'assigned': 'Virginia',
  'time': 1.0,
  'type': 'Daily'},
 {'id': 10,
  'name': 'Tidy room',
  'assigned': 'Ev